<a href="https://colab.research.google.com/github/AfraazUlHaque/Non-Linear-SVM/blob/main/house_nlsvm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


housing = fetch_california_housing(as_frame=True)
X = housing.data[['MedInc', 'HouseAge', 'AveRooms']].values

median_val = housing.target.median()
y_binary = np.where(housing.target > median_val, 1, 0)


np.random.seed(42)
indices = np.random.choice(X.shape[0], 1000, replace=False)
X_subset = X[indices]
y_subset = y_binary[indices]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_subset)

svm_model = SVC(kernel='rbf', C=4.0, gamma='scale')
svm_model.fit(X_scaled, y_subset)


x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
z_min, z_max = X_scaled[:, 2].min() - 0.5, X_scaled[:, 2].max() + 0.5

xx, yy, zz = np.meshgrid(
    np.linspace(x_min, x_max, 25),
    np.linspace(y_min, y_max, 25),
    np.linspace(z_min, z_max, 25),
)

mesh_points = np.c_[xx.ravel(), yy.ravel(), zz.ravel()]
Z = svm_model.decision_function(mesh_points)
Z = Z.reshape(xx.shape)


scatter_points = go.Scatter3d(
    x=X_scaled[:, 0],
    y=X_scaled[:, 1],
    z=X_scaled[:, 2],
    mode='markers',
    marker=dict(
        size=5,
        color=y_subset,
        colorscale='Viridis',
        opacity=0.8,
        symbol='circle',
        line=dict(color='black', width=0.5),
    ),
    name='Housing Regions',
)

decision_surface = go.Isosurface(
    x=xx.flatten(),
    y=yy.flatten(),
    z=zz.flatten(),
    value=Z.flatten(),
    isomin=0.0,
    isomax=0.0,
    surface=dict(count=1, fill=0.4, pattern='all'),
    colorscale='Spectral',
    opacity=0.35,
    name='Non-Linear Boundary (RBF)',
)


layout = go.Layout(
    title='<b>California Housing - Non-Linear SVM 3D Boundary</b>',
    scene=dict(
        xaxis_title='Scaled Median Income',
        yaxis_title='Scaled House Age',
        zaxis_title='Scaled Average Rooms',
        bgcolor='rgb(245, 245, 245)',
    ),
    margin=dict(r=0, b=0, l=0, t=40),
)

fig = go.Figure(data=[scatter_points, decision_surface], layout=layout)
fig.show()